In [23]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
import model
import config
from data import YoloPascalVOCDataset
import loss
from torchinfo import summary
import utils
from torchmetrics.detection import MeanAveragePrecision

In [24]:
train_dataset = YoloPascalVOCDataset(
    data_path="datas",
    set_type="train",
    is_augment=True
)

test_dataset = YoloPascalVOCDataset(
    data_path="datas",
    set_type="val",
    is_augment=True
)

In [25]:
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
)

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}")
model = model.YouOnlyLookOnceV1(input_size=3).to(device)

Training device: cuda


In [27]:
summary(model, input_size=(1, 3, 448, 448), col_names=["input_size", "output_size", "num_params", "trainable"], verbose=0)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
YouOnlyLookOnceV1                        [1, 3, 448, 448]          [1, 7, 7, 30]             --                        True
├─Sequential: 1-1                        [1, 3, 448, 448]          [1, 1024, 7, 7]           --                        True
│    └─Conv2d: 2-1                       [1, 3, 448, 448]          [1, 64, 224, 224]         9,472                     True
│    └─LeakyReLU: 2-2                    [1, 64, 224, 224]         [1, 64, 224, 224]         --                        --
│    └─MaxPool2d: 2-3                    [1, 64, 224, 224]         [1, 64, 112, 112]         --                        --
│    └─Conv2d: 2-4                       [1, 64, 112, 112]         [1, 192, 56, 56]          110,784                   True
│    └─LeakyReLU: 2-5                    [1, 192, 56, 56]          [1, 192, 56, 56]          --                        --
│    └─Ma

In [28]:
loss_fn = loss.SumSquaredErrorLoss()
optimizer = torch.optim.SGD(params=model.parameters(), 
                             lr=config.LEARNING_RATE, 
                             weight_decay=config.WEIGHT_DECAY, 
                             momentum=config.MOMENTUM)

In [29]:
def calculate_accuracy(preds, targets):
    pred_classes = torch.argmax(preds[..., :config.C], dim=-1)      # Tahmin edilen sınıf index
    target_classes = torch.argmax(targets[..., :config.C], dim=-1)  # Hedef sınıf index

    obj_mask = targets[..., config.C + 4] > 0                       # Hedefte nesne var mı yok mu maskesi (1: nesne var, 0: nesne yok)
    correct = (pred_classes == target_classes) & obj_mask           # Hedef ile tahmin edilen sınıfın eşleşip eşleşmediğini kontrol eder ve nesne maskesi ile çarpar. Bu, yalnızca nesne var olan hücrelerde doğru tahminleri sayar.

    acc = correct.sum().item() / obj_mask.sum().clamp(min=1)        # Doğru tahmin edilen hücrelerin sayısını, nesne var olan hücrelerin sayısına böler. clamp(min=1) ile sıfıra bölünme hatası önlenir.

    return acc.item() * 100                                              # Sonucu döndürür

In [30]:
from utils import scheduler_lambda


def train(model, epochs, loss_fn, optimizer, train_loader, test_loader, device):

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        train_acc = 0.0
        test_loss = 0.0
        test_acc = 0.0

        for param_grop in optimizer.param_groups:
            param_grop['lr'] = config.LEARNING_RATE * scheduler_lambda(epoch)

        for batch_idx, (data, labels, _) in enumerate(train_loader):
            
            data, labels = data.to(device), labels.to(device)

            pred = model(data)
            loss = loss_fn(pred, labels)

            train_loss += loss.item()
            train_acc += calculate_accuracy(pred, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)
        train_acc /= len(train_loader)

        model.eval()
        with torch.inference_mode():
            for batch_idx, (data, labels, _) in enumerate(test_loader):
                data, labels = data.to(device), labels.to(device)

                pred = model(data)
                loss = loss_fn(pred, labels)

                test_loss += loss.item()
                test_acc += calculate_accuracy(pred, labels)

            test_loss /= len(test_loader)
            test_acc /= len(test_loader)

        print(f"Epoch: {epoch}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

In [31]:
train(model, config.EPOCHS, loss_fn, optimizer, train_loader, test_loader, device)

Epoch: 0, Train Loss: 4.4312, Train Acc: 26.9269, Test Loss: 3.5273, Test Acc: 33.7039
Epoch: 1, Train Loss: 3.5900, Train Acc: 32.3420, Test Loss: 3.4998, Test Acc: 32.9437
Epoch: 2, Train Loss: 3.5580, Train Acc: 32.8878, Test Loss: 3.4857, Test Acc: 33.7193
Epoch: 3, Train Loss: 3.5310, Train Acc: 33.4147, Test Loss: 3.4540, Test Acc: 33.4360
Epoch: 4, Train Loss: 3.5306, Train Acc: 33.6853, Test Loss: 3.4629, Test Acc: 33.4426
Epoch: 5, Train Loss: 3.4964, Train Acc: 33.1643, Test Loss: 3.4778, Test Acc: 33.2862
Epoch: 6, Train Loss: 3.4852, Train Acc: 33.0214, Test Loss: 3.4099, Test Acc: 33.6191
Epoch: 7, Train Loss: 3.4904, Train Acc: 33.7099, Test Loss: 3.4336, Test Acc: 33.1744
Epoch: 8, Train Loss: 3.4737, Train Acc: 32.7356, Test Loss: 3.4065, Test Acc: 33.3830
Epoch: 9, Train Loss: 3.4567, Train Acc: 32.8838, Test Loss: 3.3910, Test Acc: 33.1104


KeyboardInterrupt: 